# Diabetes ML Training Notebook

**Pipeline:**
1. Connect to i2b2 CRC database and explore diabetes concept codes
2. Query diabetic patients → create **positive** patient set
3. Query remaining patients → create **negative** patient set
4. Create the ML concept via the ETL API
5. Submit a build-model job and poll until complete

## 0. Install dependencies (run once)

In [ ]:
# !pip install psycopg2-binary requests pandas

## 1. Configuration

In [ ]:
import psycopg2
import pandas as pd
import requests
import base64
import json
import time
import datetime

# ── Flask API ──────────────────────────────────────────────────────────────────
BASE_URL = "http://localhost:5000"
PROJECT  = "Demo"
USERNAME = "demo"
PASSWORD = "demouser"   # i2b2 password — used to get a session key

PM_SERVICE_URL = "http://localhost:8080/i2b2/services/PMService/getServices"

# ── Direct DB connection (for creating patient sets) ──────────────────────────
# Values come from deployment/pg/.env
DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "dbname":   "i2b2demodata",   # CRC DB (I2B2_DS_CRC_DB)
    "user":     "i2b2",
    "password": "demouser",
    "options":  "-c search_path=i2b2demodata"
}

# ── ML concept config ─────────────────────────────────────────────────────────
ML_CONCEPT_CODE = "diabetes_type2_ml"
ML_CONCEPT_PATH = "/ML/Diagnosis/Diabetes_Type2"

# Patient set names (must match exactly what is stored in qt_query_result_instance.description)
POSITIVE_PS_NAME = "positive_diabetes"
NEGATIVE_PS_NAME = "negative_diabetes"

# ICD-10 prefix for Type 2 Diabetes (E11.*)
DIABETES_ICD10_PREFIX = "ICD10:E11"

print("Config loaded.")

## 2. Auth — get i2b2 session key

In [ ]:
def get_session_key(pm_url, username, password):
    xml = """<?xml version=\"1.0\" encoding=\"UTF-8\"?>
<i2b2:request xmlns:i2b2=\"http://www.i2b2.org/xsd/hive/msg/1.1/\">
  <message_header>
    <security>
      <domain>i2b2demo</domain>
      <username>{u}</username>
      <password>{p}</password>
    </security>
  </message_header>
  <message_body/>
</i2b2:request>""".format(u=username, p=password)

    resp = requests.post(
        pm_url, data=xml,
        headers={"Content-Type": "application/xml", "Accept": "*/*"}
    ).text
    for part in resp.split("</password>"):
        if "SessionKey:" in part:
            return part[part.find("SessionKey:") + len("SessionKey:"):].strip()
    raise ValueError("Session key not found. Check PM_SERVICE_URL and credentials.")


def make_headers(project, username, session_key):
    creds = base64.b64encode(f"{project}\\{username}:{session_key}".encode()).decode()
    return {
        "Authorization": f"Basic {creds}",
        "X-Project-Name": project,
        "Content-Type":  "application/json",
    }


try:
    SESSION_KEY = get_session_key(PM_SERVICE_URL, USERNAME, PASSWORD)
    HEADERS = make_headers(PROJECT, USERNAME, SESSION_KEY)
    print(f"Session key obtained: {SESSION_KEY[:20]}...")
except Exception as e:
    # If PM service is not reachable from outside Docker, use password directly
    print(f"PM service unreachable ({e}). Using password as session key (works if session already exists).")
    HEADERS = make_headers(PROJECT, USERNAME, PASSWORD)

## 3. Explore — find diabetes concept codes in the database

In [ ]:
def db_connect():
    return psycopg2.connect(**DB_CONFIG)


with db_connect() as conn:
    df_concepts = pd.read_sql("""
        SELECT concept_cd, concept_path, name_char
        FROM concept_dimension
        WHERE concept_cd ILIKE %(prefix)s
        ORDER BY concept_cd
        LIMIT 30
    """, conn, params={"prefix": DIABETES_ICD10_PREFIX + "%"})

print(f"Found {len(df_concepts)} diabetes concept codes")
df_concepts

## 4. Count diabetic vs non-diabetic patients

In [ ]:
DIABETES_CODES = tuple(df_concepts["concept_cd"].tolist())

with db_connect() as conn:
    df_positive = pd.read_sql("""
        SELECT DISTINCT patient_num
        FROM observation_fact
        WHERE concept_cd = ANY(%(codes)s)
    """, conn, params={"codes": list(DIABETES_CODES)})

    df_all = pd.read_sql("""
        SELECT DISTINCT patient_num FROM patient_dimension
    """, conn)

df_negative = df_all[~df_all["patient_num"].isin(df_positive["patient_num"])]

print(f"Positive (diabetic) patients : {len(df_positive):,}")
print(f"Negative (non-diabetic) patients: {len(df_negative):,}")
print(f"Total patients                  : {len(df_all):,}")

## 5. Create patient sets in i2b2

Inserts into `QT_QUERY_MASTER → QT_QUERY_INSTANCE → QT_QUERY_RESULT_INSTANCE → QT_PATIENT_SET_COLLECTION`.  
The ML engine looks up patient sets by name using `'Patient Set for "<name>"'`.

In [ ]:
def create_patient_set(conn, query_name, patient_nums, project_id="Demo", user_id="demo"):
    """Insert a named patient set into the i2b2 QT tables."""
    now = datetime.datetime.now()
    with conn.cursor() as cur:
        # 1 — query master
        cur.execute("""
            INSERT INTO qt_query_master
                (name, user_id, group_id, create_date, delete_flag, request_xml)
            VALUES (%s, %s, %s, %s, %s, %s)
            RETURNING query_master_id
        """, (query_name, user_id, project_id, now, "N", ""))
        master_id = cur.fetchone()[0]

        # 2 — query instance
        cur.execute("""
            INSERT INTO qt_query_instance
                (query_master_id, user_id, group_id, start_date, end_date, status_type_id)
            VALUES (%s, %s, %s, %s, %s, %s)
            RETURNING query_instance_id
        """, (master_id, user_id, project_id, now, now, 3))
        instance_id = cur.fetchone()[0]

        # 3 — result instance  (description must be exactly 'Patient Set for "<name>"')
        description = f'Patient Set for "{query_name}"'
        cur.execute("""
            INSERT INTO qt_query_result_instance
                (query_instance_id, result_type_id, set_size,
                 start_date, end_date, status_type_id, description)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            RETURNING result_instance_id
        """, (instance_id, 1, len(patient_nums), now, now, 3, description))
        result_id = cur.fetchone()[0]

        # 4 — patient set members
        cur.executemany("""
            INSERT INTO qt_patient_set_collection (result_instance_id, patient_num)
            VALUES (%s, %s)
        """, [(result_id, int(p)) for p in patient_nums])

        conn.commit()
    print(f"  Created '{query_name}' → result_instance_id={result_id}  ({len(patient_nums):,} patients)")
    return result_id


with db_connect() as conn:
    # Delete stale patient sets with the same names first to avoid duplicates
    with conn.cursor() as cur:
        for name in [POSITIVE_PS_NAME, NEGATIVE_PS_NAME]:
            desc = f'Patient Set for "{name}"'
            cur.execute("""
                DELETE FROM qt_patient_set_collection
                WHERE result_instance_id IN (
                    SELECT result_instance_id FROM qt_query_result_instance
                    WHERE description = %s
                )
            """, (desc,))
            cur.execute("DELETE FROM qt_query_result_instance WHERE description = %s", (desc,))
        conn.commit()
    print("Old patient sets cleaned up.")

    pos_result_id = create_patient_set(
        conn, POSITIVE_PS_NAME,
        df_positive["patient_num"].tolist()
    )
    neg_result_id = create_patient_set(
        conn, NEGATIVE_PS_NAME,
        df_negative["patient_num"].tolist()
    )

print("\nPatient sets ready.")

## 6. Find data paths and label paths in the ontology

The ML engine uses `data_paths` (EHR feature paths) and `label_paths` (the label concept path).  
We use the diabetes concept path itself as the label.

In [ ]:
with db_connect() as conn:
    # Top-level concept paths to use as EHR features (e.g. diagnoses, vitals)
    df_top_paths = pd.read_sql("""
        SELECT DISTINCT
            split_part(concept_path, '\\', 3) AS top_path,
            COUNT(*) AS concept_count
        FROM concept_dimension
        WHERE concept_path LIKE '\\%%\\%%'
        GROUP BY 1
        ORDER BY concept_count DESC
        LIMIT 20
    """, conn)

df_top_paths

In [ ]:
# ── Choose data paths (EHR feature domains to train on) ───────────────────────
# Edit this list to include the concept tree paths relevant to your data.
# Example: diagnoses and labs. Use paths as they appear in concept_path column.
DATA_PATHS   = ["/Diagnoses", "/Demographics"]

# ── Label path — the diabetes concept branch ──────────────────────────────────
LABEL_PATHS  = ["/Diagnoses/ICD10/E11"]

print("Data paths :", DATA_PATHS)
print("Label paths:", LABEL_PATHS)

## 7. Create the ML concept via the ETL API

In [ ]:
ml_concept_payload = {
    "code":        ML_CONCEPT_CODE,
    "path":        ML_CONCEPT_PATH,
    "description": "Logistic regression model — Type 2 Diabetes prediction",
    "blob": {
        "positive_patient_set": [POSITIVE_PS_NAME],
        "negative_patient_set": [NEGATIVE_PS_NAME],
        "data_paths":           DATA_PATHS,
        "label_paths":          LABEL_PATHS,
        "data_period_start":    "1990-01-01",
        "data_period_end":      "2024-12-31",
        "test_size":            0.25,
        "sample_size_limit":    50000
    }
}

r = requests.post(
    f"{BASE_URL}/etl/ml_build_model",
    headers=HEADERS,
    json=ml_concept_payload
)

print(f"Status: {r.status_code}")
try:
    print(json.dumps(r.json(), indent=2))
except Exception:
    print(r.text)

## 8. Submit the build-model job

In [ ]:
job_payload = {
    "jobType": "ml-build",
    "input":   {"path": ML_CONCEPT_PATH}
}

r = requests.post(f"{BASE_URL}/etl/job", headers=HEADERS, json=job_payload)
print(f"Status: {r.status_code}")

try:
    job_response = r.json()
    print(json.dumps(job_response, indent=2))
    JOB_ID = job_response.get("jobId") or job_response.get("id")
    print(f"\nJob ID: {JOB_ID}")
except Exception:
    print(r.text)
    JOB_ID = None

## 9. Poll job status until complete

In [ ]:
def poll_job(job_id, interval=10, timeout=600):
    """Poll GET /etl/job until status is COMPLETED or FAILED."""
    if job_id is None:
        print("No job ID — check the response above.")
        return

    deadline = time.time() + timeout
    while time.time() < deadline:
        r = requests.get(
            f"{BASE_URL}/etl/job",
            headers=HEADERS,
            params={"jobId": job_id}
        )
        try:
            data = r.json()
        except Exception:
            print(r.text)
            break

        status = data.get("status") or data.get("jobStatus") or str(data)
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] Job {job_id} — {status}")

        if status in ("COMPLETED", "FAILED", "ERROR"):
            print(json.dumps(data, indent=2))
            return data

        time.sleep(interval)

    print("Timeout reached — job may still be running. Check manually.")


result = poll_job(JOB_ID)

## 10. Inspect the trained model stored in concept_blob

In [ ]:
with db_connect() as conn:
    df_model = pd.read_sql("""
        SELECT concept_cd, concept_path, concept_blob
        FROM concept_dimension
        WHERE concept_cd = %(code)s
    """, conn, params={"code": ML_CONCEPT_CODE})

if df_model.empty:
    print("Model concept not found yet — job may still be running.")
else:
    raw_blob = df_model.iloc[0]["concept_blob"]
    try:
        blob = json.loads(raw_blob.replace("'", '"'))
    except Exception:
        blob = raw_blob  # print raw if parse fails

    print(f"Concept path : {df_model.iloc[0]['concept_path']}")
    print()

    if isinstance(blob, dict):
        # Print metrics, skip the serialized model bytes
        for k, v in blob.items():
            if k != "serialized_model":
                print(f"  {k}: {v}")
    else:
        print(blob)

## 11. (Optional) Verify patient set counts in the DB

In [ ]:
with db_connect() as conn:
    df_sets = pd.read_sql("""
        SELECT ri.description, ri.set_size, ri.start_date
        FROM qt_query_result_instance ri
        WHERE ri.description IN (
            %(pos_desc)s,
            %(neg_desc)s
        )
        ORDER BY ri.start_date DESC
    """, conn, params={
        "pos_desc": f'Patient Set for "{POSITIVE_PS_NAME}"',
        "neg_desc": f'Patient Set for "{NEGATIVE_PS_NAME}"'
    })

df_sets